In [0]:
# Create AutoLoader input folder in volume
dbutils.fs.mkdirs("/Volumes/dev/bronze/landing/autoloader_input/2010/12/01") 
dbutils.fs.mkdirs("/Volumes/dev/bronze/landing/autoloader_input/2010/12/02") 
dbutils.fs.mkdirs("/Volumes/dev/bronze/landing/autoloader_input/2010/12/03") 
dbutils.fs.mkdirs("/Volumes/dev/bronze/landing/autoloader_input/2010/12/05") 
dbutils.fs.mkdirs("/Volumes/dev/bronze/landing/autoloader_input/2010/12/06") 
dbutils.fs.mkdirs("/Volumes/dev/bronze/landing/autoloader_input/2010/12/07") 
dbutils.fs.mkdirs("/Volumes/dev/bronze/landing/autoloader_input/2010/12/08") 

In [0]:
# Create checkpoint location in volume

dbutils.fs.mkdirs("/Volumes/dev/bronze/landing/checkpoint/autoloader")




In [0]:
# COpy files to nested locations

dbutils.fs.cp("/databricks-datasets/definitive-guide/data/retail-data/by-day/2010-12-01.csv", "/Volumes/dev/bronze/landing/autoloader_input/2010/12/01")
dbutils.fs.cp("/databricks-datasets/definitive-guide/data/retail-data/by-day/2010-12-02.csv", "/Volumes/dev/bronze/landing/autoloader_input/2010/12/02")
dbutils.fs.cp("/databricks-datasets/definitive-guide/data/retail-data/by-day/2010-12-03.csv", "/Volumes/dev/bronze/landing/autoloader_input/2010/12/03")

In [0]:
# Read files using AutoLoader with checkpoint

#   1. DIrectory listing: uses API calls to detect new files
#   2. FIle notification: uses notification and queru services

df = (spark.readStream.format("cloudFiles")
      .option("cloudFiles.format", "csv")
      .option("pathGlobFilter", "*.csv")
      .option("header", "true")
      .option("cloudFiles.schemaHints", "Quantity int, UnitPrice double")
      .option("cloudFiles.schemaLocation", "/Volumes/dev/bronze/landing/autoloader/1/")
      .load("/Volumes/dev/bronze/landing/autoloader_input/*/")
)


In [0]:
# Write data to delta table

from pyspark.sql.functions import col

(
  df
  .withColumn("__file", col("_metadata.file_name"))
  .writeStream
  .option("checkpointLocation", "/Volumes/dev/bronze/landing/checkpoint/autoloader/1/")
  .option("mergeSchema", "true")
  .outputMode("append")
  .trigger(availableNow = True)
  .toTable("dev.bronze.invoice_al_1")
)

In [0]:
%sql
select * from dev.bronze.invoice_al_1

In [0]:
%sql

select __file, count(1) from dev.bronze.invoice_al_1 group by __file

In [0]:
dbutils.fs.cp("/databricks-datasets/definitive-guide/data/retail-data/by-day/2010-12-05.csv", "/Volumes/dev/bronze/landing/autoloader_input/2010/12/05")

In [0]:
%sql
select * from dev.bronze.invoice_al_1

In [0]:
# Read files using AutoLoader with checkpoint

#   1. DIrectory listing: uses API calls to detect new files
#   2. FIle notification: uses notification and query services

df = (spark.readStream.format("cloudFiles")
      .option("cloudFiles.format", "csv")
      .option("pathGlobFilter", "*.csv")
      .option("header", "true")
      .option("cloudFiles.schemaHints", "Quantity int, UnitPrice double")
      .option("cloudFiles.schemaLocation", "/Volumes/dev/bronze/landing/autoloader/2/")
      .option("cloudFiles.schemaEvolutionMode", "rescue")
      .load("/Volumes/dev/bronze/landing/autoloader_input/*/")
)


In [0]:
# Write data to delta table

from pyspark.sql.functions import col

(
  df
  .withColumn("__file", col("_metadata.file_name"))
  .writeStream
  .option("checkpointLocation", "/Volumes/dev/bronze/landing/checkpoint/autoloader/2/")
  .option("mergeSchema", "true")
  .outputMode("append")
  .trigger(availableNow = True)
  .toTable("dev.bronze.invoice_al_2")
)

In [0]:
%sql
select * from dev.bronze.invoice_al_2

In [0]:
%sql

select __file, count(1) from dev.bronze.invoice_al_2 group by __file

# Evolution Mode  = None


In [0]:
# Read files using AutoLoader with checkpoint

#   1. DIrectory listing: uses API calls to detect new files
#   2. FIle notification: uses notification and query services

df = (spark.readStream.format("cloudFiles")
      .option("cloudFiles.format", "csv")
      .option("pathGlobFilter", "*.csv")
      .option("header", "true")
      .option("cloudFiles.schemaHints", "Quantity int, UnitPrice double")
      .option("cloudFiles.schemaLocation", "/Volumes/dev/bronze/landing/autoloader/3/")
      .option("cloudFiles.schemaEvolutionMode", "none")
      .load("/Volumes/dev/bronze/landing/autoloader_input/*/")
)


In [0]:
# Write data to delta table

from pyspark.sql.functions import col

(
  df
  .withColumn("__file", col("_metadata.file_name"))
  .writeStream
  .option("checkpointLocation", "/Volumes/dev/bronze/landing/checkpoint/autoloader/3/")
  .option("mergeSchema", "true")
  .outputMode("append")
  .trigger(availableNow = True)
  .toTable("dev.bronze.invoice_al_3")
)

In [0]:
%sql
select * from dev.bronze.invoice_al_3 where __file = "2010-12-06.csv"

In [0]:
%sql

select __file, count(1) from dev.bronze.invoice_al_3 group by __file